# Win-prob model evaluation (offline walk-forward)

Out-of-sample holdout of the v4 model straight from `data/sportsball.duckdb` (no Postgres). Reuses the production `walk_forward` + feature builder, fits on the earlier games, scores the later ones — so these are **generalization** numbers, not in-sample fit.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'src'/'sportsball').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src')); sys.path.insert(0, str(ROOT/'scripts'))
from train_eval_duckdb import load_events, build_market_pit, _pipeline, K, HFA, DEFAULT_DB
from sportsball.pipelines._elo import walk_forward
from sportsball.quant import features as feat

In [ ]:
rows_raw = load_events(str(DEFAULT_DB))
market_pit = build_market_pit(rows_raw)
results = [(d,h,a,hs,as_) for (d,h,a,hs,as_,_hc,_ac) in rows_raw]
frows, snaps = walk_forward(results, K, HFA, mov_enabled=True, carry=0.75,
                            gap_days=90, form_window=10, market_pit=market_pit)
X = np.array([r.features for r in frows]); y = np.array([1 if r.actual>=1 else 0 for r in frows])
split = 0.85; cut = int(len(X)*split)
model = _pipeline().fit(X[:cut], y[:cut])
p = model.predict_proba(X[cut:])[:,1]; yte = y[cut:]
print(f'{len(X)} games loaded | train {cut} | test {len(yte)} | home-win base {yte.mean():.3f}')

## Headline metrics

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
print(f'log_loss {log_loss(yte,p,labels=[0,1]):.4f} | brier {brier_score_loss(yte,p):.4f} | '
      f'accuracy {accuracy_score(yte,(p>=0.5).astype(int)):.4f}')

## Reliability / calibration curve

On the diagonal = well-calibrated.

In [ ]:
from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(yte, p, n_bins=10, strategy='quantile')
plt.figure(figsize=(6,6))
plt.plot([0,1],[0,1],'k--', label='perfect')
plt.plot(mean_pred, frac_pos, 'o-', label='v4 model')
plt.xlabel('predicted P(home win)'); plt.ylabel('observed frequency')
plt.title('Reliability (holdout)'); plt.legend(); plt.show()

## Accuracy by confidence

How much better than a coin flip when the model is confident?

In [ ]:
df = pd.DataFrame({'p': p, 'y': yte})
df['conf'] = (df['p']-0.5).abs()
df['bucket'] = pd.cut(df['conf'], bins=[0,0.05,0.1,0.15,0.2,0.5])
g = df.groupby('bucket', observed=True).apply(
    lambda d: pd.Series({'n': len(d), 'accuracy': ((d.p>=0.5).astype(int)==d.y).mean()}))
g

## Model vs market (CLV proxy)

The model's P(home) against the no-vig market probability on lined test games. Points far off the diagonal are where the model disagrees with the close — the raw material for (and risk of) edge.

In [ ]:
mkt = feat.FEATURE_ORDER.index('market_logit')
mlogit = X[cut:, mkt]; mask = mlogit != 0
mkt_prob = 1/(1+np.exp(-mlogit[mask]))
plt.figure(figsize=(6,6))
plt.scatter(mkt_prob, p[mask], s=8, alpha=0.25)
plt.plot([0,1],[0,1],'k--')
plt.xlabel('market no-vig P(home)'); plt.ylabel('model P(home)')
plt.title(f'Model vs market — {int(mask.sum())} lined test games'); plt.show()